We constructed K562-LenMatch dataset in which the length distribution of
the negative samples was adjusted to match that of the positive samples. Specifically, from the K562 training set we retained
all positive sequences shorter than 512 bp and generated the same number of negative
samples by randomly sampling contiguous subsequences of matching length from
longer negative sequences in K562 so that their length distribution closely followed
that of the positive sequences.
1. run this script for both sets of source files: K562/validation and K562/train
2. use notebook "dataset_creation_k562.ipynb" for train result file for spliting it to dev and train  and extracting sequences 
3. use notebook "dataset_creation-K562_testset.ipynb" for validation result file extracting sequences for unseen test set 

In [2]:
# Balanced NEG generation: fully match POS length distribution using neg_all as donors
from __future__ import annotations
from pathlib import Path
from typing import List, Tuple, Dict
from collections import Counter
import random
import math

# ---- Config ----
POS_BED = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/Ind_pos.bed")
NEG_BED = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/Ind_neg.bed")
OUTDIR  = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/K562_C_Rand_Trnc")
RANDOM_SEED = 42
TARGET_BUCKETS = [99, 199, 299, 399, 499]  # desired exact lengths
MAXLEN = 512

rng = random.Random(RANDOM_SEED)

# ---- Types & I/O helpers ----
Chrom, Start, End = str, int, int
Interval = Tuple[Chrom, Start, End]

def read_bed_first3(path: Path) -> List[Interval]:
    ivals: List[Interval] = []
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if not line.strip() or line.startswith(("track","browser","#")):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            try:
                st = int(parts[1]); en = int(parts[2])
            except ValueError:
                continue
            if en < st:
                continue
            ivals.append((parts[0], st, en))
    return ivals

def write_bed(path: Path, ivs: List[Interval]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as out:
        for c,s,e in ivs:
            out.write(f"{c}\t{s}\t{e}\n")

def iv_len(iv: Interval) -> int:
    return iv[2] - iv[1]

def nearest_bucket(length: int, buckets: List[int]) -> int:
    # ties go to smaller bucket
    return min(buckets, key=lambda b: (abs(length - b), b))

def truncate_to_length(
    iv: Interval,
    target_len: int,
    rng: random.Random,
) -> Interval:
    c, s, e = iv
    length = e - s

    target_len = min(target_len, MAXLEN)
    if length <= target_len:
        return iv

    max_offset = length - target_len
    offset = rng.randint(0, max_offset)

    return (c, s + offset, s + offset + target_len)




def summarize_bucket(name: str, ivs: List[Interval]) -> str:
    cnts = Counter(nearest_bucket(iv_len(iv), TARGET_BUCKETS) for iv in ivs)
    tot = len(ivs)
    parts = [f"{b}:{cnts.get(b,0)}({(cnts.get(b,0)/tot):.1%})" if tot else f"{b}:0(0.0%)" for b in TARGET_BUCKETS]
    return f"{name} total={tot} | " + " ".join(parts)

def available_candidates(min_len: int) -> List[int]:
    """Return indices of donors not used with length >= min_len."""
    return [i for i,iv in enumerate(neg_donors) if i not in used and iv_len(iv) >= min_len]

    
# ---- Load ----
assert POS_BED.exists(), f"Missing {POS_BED}"
assert NEG_BED.exists(), f"Missing {NEG_BED}"

pos_all = read_bed_first3(POS_BED)
neg_all = read_bed_first3(NEG_BED)

# POS basis: keep <512 and derive bucket targets
pos_lt = [iv for iv in pos_all if iv_len(iv) < MAXLEN]
pos_bucket_counts = Counter(nearest_bucket(iv_len(iv), TARGET_BUCKETS) for iv in pos_lt)

# NEG donors: use ALL neg_all (both <512 and >=512), but each donor can be used once
neg_donors = list(neg_all)
rng.shuffle(neg_donors)  # avoid positional bias


used = set()               # indices in neg_donors already taken
neg_selected: List[Interval] = []



# Greedy allocation: process buckets descending so longer requirements get priority
unfilled: Dict[int, int] = {}

for b in sorted(TARGET_BUCKETS, reverse=True):
    need_b = pos_bucket_counts.get(b, 0)
    if need_b == 0:
        continue
    cand_idx = available_candidates(min_len=b)
    if len(cand_idx) >= need_b:
        take_idx = rng.sample(cand_idx, k=need_b)
    else:
        take_idx = cand_idx
        unfilled[b] = need_b - len(cand_idx)
        print(f"Warning: bucket {b} needs {need_b}, but only {len(cand_idx)} donors with len>= {b} available.")
    for i in take_idx:
        used.add(i)
        neg_selected.append(truncate_to_length(neg_donors[i], b, rng))

# Final sets
pos_out_ivs = pos_lt
neg_out_ivs = neg_selected

# ---- Save & report ----
OUTDIR.mkdir(parents=True, exist_ok=True)
pos_out = OUTDIR / "pos_lt512.bed"
neg_out = OUTDIR / "neg_lt512.bed"
write_bed(pos_out, pos_out_ivs)
write_bed(neg_out, neg_out_ivs)

print(summarize_bucket("POS(<512)", pos_out_ivs))
print(summarize_bucket("NEG(matched)", neg_out_ivs))
if unfilled:
    missing_total = sum(unfilled.values())
    print("Unfilled buckets:", unfilled, "| total deficit:", missing_total)
    print("Resulting NEG count is smaller than POS; consider relaxing targets or using a larger NEG pool.")
print(f"Wrote:\n  {pos_out}\n  {neg_out}")


POS(<512) total=16031 | 99:3721(23.2%) 199:4555(28.4%) 299:3761(23.5%) 399:2531(15.8%) 499:1463(9.1%)
NEG(matched) total=16031 | 99:3721(23.2%) 199:4555(28.4%) 299:3761(23.5%) 399:2531(15.8%) 499:1463(9.1%)
Wrote:
  /p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/K562_C_Rand_Trnc/pos_lt512.bed
  /p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/K562_C_Rand_Trnc/neg_lt512.bed


gfg